In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))

# 04 — Financial Graph Construction (Phase 4)

Money-flow graph from `artifacts/transaction_features.csv` **train-period edges only** (leakage-safe, `config.SPLIT`).
Uses `src/graph_builder.py`: `build_graph(df, train_end)`, `graph_features(df, G, label_df)`, `scc_map(G)`, `build_node_table(G, df)`, `export_graphsage_ready(G, node_table, out_dir)` and `src/config.py` (`COLS`/`SPLIT`).

Sections:
- (1) Load features + fraud-intelligence cols (`rule_score`, `anomaly_score`) if present + schema/file checks
- (2) Scale profile (nodes/edges/memory) + bounded `networkx` strategy
- (3) `build_graph` on train edges only + `build_node_table` (net_flow, flow_ratio, pass_through, fan-in/out, SCC sizes, phase-3 aggregates)
- (4) Cycle stats: bounded 2-hop `cycle_flag` + SCC size distribution (no exhaustive enumeration)
- (5) Investigation candidate subgraphs: seeds + 1–2 hop neighbours (counts only)
- (6) Centrality: pagerank / degree top (masked IDs)
- (7) Data-quality checks + `reports/graph_data_quality_report.md` + `reports/financial_graph_report.md` (masked IDs)
- (8) `export_graphsage_ready` to `artifacts/graphsage_ready/` + `graph_nodes.csv`, `graph_edges.csv`, `node_features.csv`

> Privacy: account IDs are masked (`first 8 chars + ***`) in all printed/reported outputs.
> Scale: bounded neighbourhoods (cap successors / 2-hop sets); no exhaustive cycle enumeration.

In [ ]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('04_graph')
try:
    display
except NameError:
    def display(x=None, *a, **k):
        try:
            print(x.to_string() if hasattr(x, 'to_string') else x)
        except Exception:
            print(x)

from src import config
from src.graph_builder import (
    build_graph, graph_features, scc_map, build_node_table, export_graphsage_ready,
)

C = config.COLS
SPLIT = config.SPLIT
TRAIN_END = SPLIT['train_end']
log.info('COLS=%s SPLIT=%s', C, SPLIT)

ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
FEAT = ROOT / 'artifacts' / 'transaction_features.csv'
FI = ROOT / 'artifacts' / 'fraud_intelligence_features.csv'
REPORTS = ROOT / 'reports'
GS_OUT = ROOT / 'artifacts' / 'graphsage_ready'
REPORTS.mkdir(parents=True, exist_ok=True)

def mask_id(x, k=8):
    s = str(x)
    return (s[:k] + '***') if len(s) > k else (s + '***')

# (1) Load artifacts/transaction_features.csv + fraud-intelligence cols if present + schema/file checks
try:
    assert FEAT.exists(), f'missing {FEAT}'
    log.info('found %s (%.2f MB)', FEAT, FEAT.stat().st_size / 1e6)
    df = pd.read_csv(FEAT)
    log.info('transaction_features shape=%s cols=%s', df.shape, list(df.columns)[:20])
except Exception:
    log.exception('failed to load transaction_features.csv')
    raise

# Required schema from config.COLS
try:
    need = [C['account_id'], C['counterparty'], C['transaction_id'], C['timestamp'], C['amount']]
    missing = [c for c in need if c not in df.columns]
    assert not missing, f'missing required cols {missing} have={list(df.columns)}'
    assert len(df) > 0, 'transaction_features.csv is empty'
    assert df[C['transaction_id']].notna().all(), 'null transaction_id found'
    log.info('schema check OK; rows=%d unique_txns=%d', len(df), df[C['transaction_id']].nunique())
except Exception:
    log.exception('schema check failed')
    raise

# Fraud-intelligence cols (rule_score, anomaly_score) — merge if file exists else skip+log
try:
    for c in ['rule_score', 'anomaly_score']:
        if c not in df.columns:
            log.warning('col %s not in transaction_features.csv', c)
    if FI.exists():
        log.info('found fraud-intelligence %s (%.2f MB)', FI, FI.stat().st_size / 1e6)
        fi = pd.read_csv(FI)
        key = C['transaction_id']
        use = [k for k in [key, 'rule_score', 'anomaly_score', 'rule_score_weighted'] if k in fi.columns]
        log.info('fraud-intelligence cols=%s using=%s shape=%s', list(fi.columns), use, fi.shape)
        if key in fi.columns and len(use) > 1:
            add = [u for u in use if u not in df.columns]
            if add:
                df = df.merge(fi[[key] + add], on=key, how='left')
                log.info('merged fraud-intelligence cols %s', add)
            else:
                log.info('fraud-intelligence cols already present, skip merge')
        else:
            log.warning('fraud-intelligence file lacks key %s; skipping merge', key)
    else:
        log.warning('SKIP fraud-intelligence merge: %s not found; continuing without it', FI)
    log.info('post-merge cols has rule_score=%s anomaly_score=%s rule_score_weighted=%s',
             'rule_score' in df.columns, 'anomaly_score' in df.columns, 'rule_score_weighted' in df.columns)
except Exception:
    log.exception('fraud-intelligence merge failed')
    raise

In [ ]:
# (2) Scale profile: nodes/edges/memory log + strategy selection (networkx, bounded)
try:
    a = df[C['account_id']].astype(str)
    b = df[C['counterparty']].astype(str)
    n_nodes_est = pd.concat([a, b]).nunique()
    n_edge_rows = len(df)
    n_edge_pairs = df[[C['account_id'], C['counterparty']]].astype(str).drop_duplicates().shape[0]
    mem_mb = float(df.memory_usage(deep=True).sum() / 1e6)
    log.info('SCALE profile: est_nodes=%d edge_rows=%d unique_pairs=%d df_memory=%.1f MB',
             n_nodes_est, n_edge_rows, n_edge_pairs, mem_mb)
    ts = pd.to_datetime(df[C['timestamp']], errors='coerce')
    log.info('date range %s -> %s train_end=%s', ts.min(), ts.max(), TRAIN_END)
    log.info('STRATEGY: networkx.DiGraph, aggregated (src,dst)->sum(|amount|), train-period edges only; '
             'bounded 2-hop neighbourhoods (cap 50 successors/node, 200 2-hop ids); '
             'no exhaustive simple-cycle enumeration (NP-hard); cycle_flag = self-loop or 2-hop return only; '
             'SCC via networkx.strongly_connected_components (linear time).')
    STRATEGY = {'backend': 'networkx', 'bounded': True, 'succ_cap': 50, 'two_hop_cap': 200, 'exhaustive_cycles': False}
except Exception:
    log.exception('(2) scale profile failed')
    raise

In [ ]:
# (3) build_graph on train-period edges only + node table via build_node_table
try:
    G = build_graph(df, train_end=TRAIN_END)
    log.info('built graph nodes=%d edges=%d', G.number_of_nodes(), G.number_of_edges())
except Exception:
    log.exception('(3) build_graph failed')
    raise

try:
    # Node table: money-flow (net_flow, flow_ratio, pass_through_score), fan-in/out via graph_features,
    # SCC sizes via scc_map, phase-3 aggregates (avg/max rule+anomaly)
    node_table = build_node_table(G, df)
    log.info('node_table shape=%s cols=%s', node_table.shape, list(node_table.columns))
    # Enrich with fan-in/out + cycle_flag + pagerank-style globals from graph_features (row-level -> node-level max)
    try:
        gf = graph_features(df, G)
        extra = gf.groupby(gf[C['account_id']].astype(str)).agg(
            fan_in_7d_max=('fan_in_7d', 'max'), fan_out_7d_max=('fan_out_7d', 'max'),
            cycle_flag_max=('cycle_flag', 'max'), pagerank_max=('pagerank', 'max'))
        node_table = node_table.merge(extra, left_on='node_id', right_index=True, how='left')
        log.info('enriched node_table with fan_in/out, cycle_flag, pagerank')
    except Exception:
        log.exception('graph_features enrichment failed (non-fatal); continuing')
    display(node_table.head(5))
    print(node_table.describe(include='all').to_string())
except Exception:
    log.exception('(3) build_node_table failed')
    raise

In [ ]:
# (4) Cycle stats: bounded 2-hop cycle_flag distribution + SCC size distribution (documented, non-exhaustive)
try:
    log.info('STRATEGY (4): cycle_flag(n)=1 if self-loop OR exists x in succ(n,cap=50) with edge x->n; '
             'no nx.simple_cycles / exhaustive enumeration (exponential blow-up).')
    cyc_col = 'cycle_flag_max' if 'cycle_flag_max' in node_table.columns else None
    if cyc_col:
        vc = node_table[cyc_col].fillna(0).astype(int).value_counts(dropna=False)
        print('bounded 2-hop cycle_flag distribution:')
        print(vc.to_string())
        log.info('cycle_flag dist %s', vc.to_dict())
    else:
        # fallback: compute directly bounded
        flags = {}
        for n in list(G.nodes()):
            try:
                s1 = list(G.successors(n))[:50]
            except Exception:
                s1 = []
            c = 1 if G.has_edge(n, n) else 0
            if not c:
                for x in s1:
                    if G.has_edge(x, n):
                        c = 1
                        break
            flags[n] = c
        s = pd.Series(flags)
        print('bounded 2-hop cycle_flag distribution:')
        print(s.value_counts().to_string())

    smap = scc_map(G)
    scc_sizes = pd.Series([sz for (_, sz) in smap.values()]) if smap else pd.Series([1])
    print('\nSCC size distribution (nodes sharing an SCC):')
    print(scc_sizes.describe().to_string())
    print(scc_sizes.value_counts().head(10).to_string())
    log.info('n_scc=%d max_scc=%d singleton_rate=%.3f',
             node_table['scc_id'].nunique() if 'scc_id' in node_table.columns else -1,
             int(scc_sizes.max()), float((scc_sizes == 1).mean()))
except Exception:
    log.exception('(4) cycle stats failed')
    raise

In [ ]:
# (5) Investigation candidate subgraphs: seeds (rule_score_weighted>=3 or anomaly flag), 1-2 hop neighbours, counts only
try:
    key = C['transaction_id']
    seed_mask = pd.Series(False, index=df.index)
    if 'rule_score_weighted' in df.columns:
        seed_mask |= (pd.to_numeric(df['rule_score_weighted'], errors='coerce').fillna(0) >= 3)
    elif 'rule_score' in df.columns:
        seed_mask |= (pd.to_numeric(df['rule_score'], errors='coerce').fillna(0) >= 3)
        log.warning('rule_score_weighted missing; fell back to rule_score>=3')
    else:
        log.warning('no rule score col; seed uses anomaly/label only')
    if 'anomaly_score' in df.columns:
        try:
            thr = float(df['anomaly_score'].quantile(0.95))
            seed_mask |= (pd.to_numeric(df['anomaly_score'], errors='coerce').fillna(0) >= thr)
            log.info('anomaly flag threshold p95=%.4f', thr)
        except Exception:
            log.exception('anomaly threshold failed (non-fatal)')
    if C['label'] in df.columns:
        seed_mask |= (pd.to_numeric(df[C['label']], errors='coerce').fillna(0) == 1)
    seeds = df.loc[seed_mask]
    log.info('seed txns=%d / %d (%.2%%)', len(seeds), len(df), 100 * len(seeds) / max(len(df), 1))

    UG = G.to_undirected(as_view=True)
    seed_nodes = set(seeds[C['account_id']].astype(str)) | set(seeds[C['counterparty']].astype(str))
    seed_nodes &= set(G.nodes())
    log.info('seed nodes in graph=%d', len(seed_nodes))

    # 1-2 hop neighbour counts only (bounded; no edge dump for privacy/scale)
    rows = []
    for n in list(seed_nodes)[:5000]:
        try:
            h1 = set(UG.neighbors(n)) if UG.has_node(n) else set()
            h1.discard(n)
            h2 = set()
            for x in list(h1)[:50]:
                try:
                    for y in UG.neighbors(x):
                        h2.add(y)
                        if len(h2) >= 500:
                            break
                except Exception:
                    continue
                if len(h2) >= 500:
                    break
            h2 -= h1
            h2.discard(n)
            rows.append({'seed_masked': mask_id(n), 'n_1hop': len(h1), 'n_2hop': len(h2)})
        except Exception:
            continue
    cand = pd.DataFrame(rows)
    if len(cand):
        print('candidate subgraph sizes (counts only, masked seeds):')
        display(cand.head(10))
        print(cand[['n_1hop', 'n_2hop']].describe().to_string())
        log.info('candidates summarized n=%d mean_1hop=%.1f mean_2hop=%.1f', len(cand), cand.n_1hop.mean(), cand.n_2hop.mean())
    else:
        log.warning('no candidate subgraphs (no seeds in graph)')
except Exception:
    log.exception('(5) candidate subgraphs failed')
    raise

In [ ]:
# (6) Centrality: pagerank / degree top (masked IDs)
try:
    import networkx as nx
    try:
        pr = nx.pagerank(G, weight='weight')
    except Exception:
        log.exception('pagerank failed; using zeros')
        pr = {n: 0.0 for n in G.nodes()}
    deg = dict(G.degree())
    cent = pd.DataFrame({'node_masked': [mask_id(n) for n in G.nodes()],
                           'pagerank': [float(pr.get(n, 0.0)) for n in G.nodes()],
                           'degree': [int(deg.get(n, 0)) for n in G.nodes()]})
    print('Top-10 by pagerank (masked):')
    display(cent.sort_values('pagerank', ascending=False).head(10))
    print('Top-10 by degree (masked):')
    display(cent.sort_values('degree', ascending=False).head(10))
    log.info('centrality computed nodes=%d', len(cent))
except Exception:
    log.exception('(6) centrality failed')
    raise

In [ ]:
# (7) Data-quality checks + save reports/graph_data_quality_report.md and reports/financial_graph_report.md (masked IDs)
try:
    issues = []
    # missing nodes: counterparties/accounts referenced but absent (should be 0 since build_graph adds all)
    nodes = set(G.nodes())
    miss_a = set(df[C['account_id']].astype(str)) - nodes
    miss_c = set(df[C['counterparty']].astype(str)) - nodes
    # self-loops
    try:
        n_self = int(nx.number_of_selfloops(G))
    except Exception:
        n_self = int(sum(1 for n in G.nodes() if G.has_edge(n, n)))
    # dup IDs
    n_dup_txn = int(len(df) - df[C['transaction_id']].nunique())
    # invalid amounts
    amt = pd.to_numeric(df[C['amount']], errors='coerce')
    n_bad_amt = int(amt.isna().sum() + np.isinf(amt.fillna(0)).sum())
    issues += [f'missing account nodes: {len(miss_a)}', f'missing counterparty nodes: {len(miss_c)}',
               f'self-loops: {n_self}', f'duplicate transaction_ids: {n_dup_txn}', f'invalid amounts: {n_bad_amt}']
    for i in issues:
        log.info('DQ: %s', i)
    print('\n'.join(issues))

    top_pr = cent.sort_values('pagerank', ascending=False).head(5)['node_masked'].tolist() if 'cent' in dir() else []
    dq_md = ('# Graph Data-Quality Report\n\n'
             f'- source: `{FEAT.name}` rows={len(df)} train_end={TRAIN_END}\n'
             f'- graph: nodes={G.number_of_nodes()} edges={G.number_of_edges()}\n'
             + ''.join(f'- {i}\n' for i in issues)
             + f'- fraud-intelligence file present: {FI.exists()}\n'
             + '- IDs masked (first 8 chars + ***)\n')
    (REPORTS / 'graph_data_quality_report.md').write_text(dq_md)
    log.info('wrote %s', REPORTS / 'graph_data_quality_report.md')

    cyc_line = node_table['cycle_flag_max'].value_counts().to_dict() if 'cycle_flag_max' in node_table.columns else {}
    fin_md = ('# Financial Graph Report (Phase 4)\n\n'
              f'- train_end={TRAIN_END} nodes={G.number_of_nodes()} edges={G.number_of_edges()}\n'
              f'- strategy: networkx bounded (succ_cap=50, two_hop_cap=200); no exhaustive cycle enumeration\n'
              f'- cycle_flag dist: {cyc_line}\n'
              f'- max SCC size: {int(node_table["scc_size"].max()) if "scc_size" in node_table.columns else "n/a"}\n'
              f'- top pagerank (masked): {top_pr}\n'
              f'- DQ: {"; ".join(issues)}\n'
              f'- exports: artifacts/graphsage_ready/, graph_nodes.csv, graph_edges.csv, node_features.csv\n')
    (REPORTS / 'financial_graph_report.md').write_text(fin_md)
    log.info('wrote %s', REPORTS / 'financial_graph_report.md')
except Exception:
    log.exception('(7) data-quality/reports failed')
    raise

In [ ]:
# (8) export_graphsage_ready to artifacts/graphsage_ready/ + graph_nodes/edges/node_features CSVs
try:
    info = export_graphsage_ready(G, node_table, GS_OUT)
    log.info('graphsage export %s', info)
    assert (GS_OUT / 'node_features.joblib').exists()
    assert (GS_OUT / 'edge_index.npy').exists()
    assert (GS_OUT / 'node_mapping.json').exists()
    ei = np.load(GS_OUT / 'edge_index.npy')
    log.info('edge_index shape=%s', ei.shape)
except Exception:
    log.exception('(8) export_graphsage_ready failed')
    raise

try:
    # node_features.csv: numeric matrix aligned to node_table order used in export (reindex inside export is G order)
    import joblib
    bundle = joblib.load(GS_OUT / 'node_features.joblib')
    feat_names = bundle['feature_names']
    mapping = json.loads((GS_OUT / 'node_mapping.json').read_text())
    inv = {v: k for k, v in mapping.items()}
    mat = bundle['node_features']
    nf = pd.DataFrame(mat, columns=feat_names)
    nf.insert(0, 'node_id', [inv[i] for i in range(len(inv))])
    nf.to_csv(ROOT / 'artifacts' / 'node_features.csv', index=False)
    node_table.to_csv(ROOT / 'artifacts' / 'graph_nodes.csv', index=False)
    edf = pd.DataFrame([{'src': a, 'dst': b, 'weight': d.get('weight', 0.0)} for a, b, d in G.edges(data=True)] )
    edf.to_csv(ROOT / 'artifacts' / 'graph_edges.csv', index=False)
    log.info('wrote node_features.csv %s graph_nodes.csv %s graph_edges.csv %s',
             nf.shape, node_table.shape, edf.shape)
except Exception:
    log.exception('(8) CSV exports failed')
    raise

print('PHASE 4 COMPLETE — graph built, reports + GraphSAGE-ready artifacts saved. NEXT PHASE: 05_graphsage_graph_intelligence.ipynb')